In [1]:
from dataset import load_snli_dataset, convert_to_dataframe
from tokenizer import train_tokenizer
from encoders import TokenIndicator, PairEncoder, BagofWords, MPNetEncoder
from sentence_transformers import SentenceTransformer
import numpy as np
import loglin
import cupy as cp


# 1 if compatible with GPU acceleration
print(cp.cuda.runtime.getDeviceCount())

1


In [2]:
dataset = load_snli_dataset()

In [3]:
train_df, validation_df, test_df = convert_to_dataframe(dataset)

In [4]:
corpus = list(set(train_df['premise'])) + list(set(train_df['hypothesis']))
#corpus

In [5]:
tokenizer = train_tokenizer(corpus, vocab_size=1000, min_frequency=2)

In [6]:
tokenizer.get_vocab_size()

1000

In [7]:
tokenizer.encode(corpus[0]).ids

[267,
 131,
 72,
 565,
 378,
 150,
 89,
 595,
 305,
 157,
 80,
 159,
 104,
 971,
 352,
 179,
 305,
 80,
 111,
 246,
 126,
 104,
 134,
 838]

In [8]:
# Token Indicator
encoder_tokenindicator = TokenIndicator(tokenizer)
encoding_tokenindicator = encoder_tokenindicator(corpus[0])
encoding_tokenindicator

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [9]:
# Bag of Words
encoder_bagofwords = BagofWords(tokenizer)
encoding_bagofwords = encoder_bagofwords(corpus[0])
encoding_bagofwords

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 2., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [10]:
row = train_df.iloc[0]
pair = ()

In [11]:
train_data = [((p, h), y) for p, h, y in zip(train_df["premise"], train_df["hypothesis"], train_df["label"])]
val_texts = list(zip(validation_df["premise"], validation_df["hypothesis"]))

In [12]:
# Sentence Transformer Embeddings
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from pathlib import Path

sentence_model = SentenceTransformer(
    "sentence-transformers/nli-mpnet-base-v2",
    device="cuda",
)

# Encoder for the (premise, hiphotesis) pairs returned in batches
# Generated this alternative to PairEncoder because of efficiency
def encode_pairs(data, batch_size=256):
    # data contains (premise, hypothesis) pairs
    premises = [pair[0] for pair in data]
    hypotheses = [pair[1] for pair in data]

    # Encode all sentences in batches on GPU
    premise_vectors = sentence_model.encode(
        premises,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    hypothesis_vectors = sentence_model.encode(
        hypotheses,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    return np.concatenate(
        [premise_vectors, hypothesis_vectors],
        axis=1,
    )

def load_or_encode(filename, data):
    if Path(filename).exists():
        return np.load(filename)

    embeddings = encode_pairs(data)
    np.save(filename, embeddings)
    return embeddings

train_pairs = [text for text, label in train_data]
X_train = load_or_encode("X_train.npy", train_pairs)
print(X_train.shape)

y_train = np.array([label for text, label in train_data])

X_val = load_or_encode("X_val.npy", val_texts)
print(X_val.shape)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(549367, 1536)
(9842, 1536)


In [13]:
# Commented, this is for CPU

#model = LogisticRegression(
#    solver="saga",
#    max_iter=100,
#)
#model.fit(X_train, y_train)

#preds = model.predict(X_val)

#print(accuracy_score(validation_df["label"], preds))

In [14]:
from cuml.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Transfer embeddings to GPU
X_train_gpu = cp.asarray(X_train, dtype=cp.float32)
y_train_gpu = cp.asarray(y_train)

X_val_gpu = cp.asarray(X_val, dtype=cp.float32)

# Train logistic regression on GPU
model = LogisticRegression(
    max_iter=1000,
)

model.fit(X_train_gpu, y_train_gpu)

# Predict on GPU
preds_gpu = model.predict(X_val_gpu)

# Convert predictions back to NumPy
preds = cp.asnumpy(preds_gpu)

print(accuracy_score(validation_df["label"], preds))

0.6621621621621622


In [15]:
pe = PairEncoder(TokenIndicator(tokenizer))
print("done pe")
model = loglin.fit(train_data, pe, max_iter=100)
print("done model")
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

done pe
Training on GPU
[2026-11-03 17:07:19.694] [CUML] [warning] L-BFGS: max iterations reached
[2026-11-03 17:07:19.699] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
done model
0.6190814875025401


In [16]:
pe = PairEncoder(TokenIndicator(tokenizer), overlap=True)
model = loglin.fit(train_data, pe, max_iter=100)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

Training on GPU
[2026-11-03 17:12:37.162] [CUML] [warning] L-BFGS: max iterations reached
[2026-11-03 17:12:37.162] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
0.69731761837025


In [17]:
pe = PairEncoder(BagofWords(tokenizer))
model = loglin.fit(train_data, pe, max_iter=100)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

Training on GPU
[2026-11-03 17:17:52.429] [CUML] [warning] L-BFGS: max iterations reached
[2026-11-03 17:17:52.429] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
0.6190814875025401


In [18]:
pe = PairEncoder(BagofWords(tokenizer), overlap=True)
model = loglin.fit(train_data, pe, max_iter=100)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

Training on GPU
[2026-11-03 17:23:13.538] [CUML] [warning] L-BFGS: max iterations reached
[2026-11-03 17:23:13.538] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
0.6935582198740093
